# Travel 纸面 8K → 16K 受约束超分

这个 notebook 只处理**行政区、陆地色和墨线合成前的纯纸面** `world-paper.png`。不要上传最终的 `world-parchment-v1.png/webp`，否则 AI 会把行政线一起处理。

处理方法：

1. RealESRGAN_x2plus 分块生成 16K 候选；
2. 原始 alpha 撕边单独用 Lanczos 放大；
3. 只提取 AI 候选相对于自身低频版本的高频残差；
4. 把残差叠加到原始 8K 的 Lanczos 2× 基线上；
5. 缩回 8K，量化检查 MAE、PSNR、SSIM 和低频色差。

在 Colab 中先选择 **Runtime → Change runtime type → T4 GPU**，然后按顺序运行全部单元格。16K PNG 可能超过 100 MB，Google Drive 模式通常比浏览器直接下载稳定。

In [ ]:
# @title 参数
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_INPUT_PATH = '/content/drive/MyDrive/world-paper.png'  # @param {type:"string"}
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/travel-paper-superres'  # @param {type:"string"}
DETAIL_GAIN = 0.55  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
TILE_SIZE = 256  # @param {type:"integer"}
TILE_PAD = 32  # @param {type:"integer"}
SAVE_RAW_AI = False  # @param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = True  # @param {type:"boolean"}

EXPECTED_SIZE = (8192, 4096)
UPSCALE = 2
OUTPUT_NAME = 'world-paper-16k-residual-v1.png'


In [ ]:
# @title 安装 Real-ESRGAN
from pathlib import Path
import importlib.util
import subprocess
import sys
import urllib.request

repo_parent = Path('/content') if Path('/content').exists() else Path.cwd()
repo_dir = repo_parent / 'Real-ESRGAN'
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/xinntao/Real-ESRGAN.git', str(repo_dir)
    ], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'basicsr', 'facexlib', 'gfpgan', 'scikit-image'
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir)
], check=True)

# 新版 torchvision 删除了 functional_tensor；只修正 BasicSR 的旧导入路径。
basicsr_spec = importlib.util.find_spec('basicsr')
if basicsr_spec is None or not basicsr_spec.submodule_search_locations:
    raise RuntimeError('BasicSR 安装失败')
degradations = Path(next(iter(basicsr_spec.submodule_search_locations))) / 'data' / 'degradations.py'
source = degradations.read_text(encoding='utf-8')
legacy_import = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
current_import = 'from torchvision.transforms.functional import rgb_to_grayscale'
if legacy_import in source:
    degradations.write_text(source.replace(legacy_import, current_import), encoding='utf-8')

# editable install 在部分托管 notebook kernel 中不会立即刷新 import path。
# 直接加入仓库根目录，并在进入推理前完成一次真实导入检查。
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
importlib.invalidate_caches()
try:
    import realesrgan
except ModuleNotFoundError as exc:
    raise RuntimeError(
        f'Real-ESRGAN 安装后仍无法导入；仓库目录为 {repo_dir}，请保留本单元格完整输出'
    ) from exc

model_dir = repo_dir / 'weights'
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / 'RealESRGAN_x2plus.pth'
if not model_path.exists():
    urllib.request.urlretrieve(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
        model_path
    )
print(f'Real-ESRGAN import: {Path(realesrgan.__file__).resolve()}')
print(f'Model ready: {model_path}')


In [ ]:
# @title 上传纯纸面并准备输出目录
from pathlib import Path
from PIL import Image

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    input_path = Path(DRIVE_INPUT_PATH)
    output_dir = Path(DRIVE_OUTPUT_DIR)
else:
    from google.colab import files
    default_input = Path('/content/world-paper.png')
    if not default_input.exists():
        uploaded = files.upload()
        png_names = [name for name in uploaded if name.lower().endswith('.png')]
        if len(png_names) != 1:
            raise RuntimeError('请只上传一个 world-paper.png')
        input_path = Path('/content') / png_names[0]
    else:
        input_path = default_input
    output_dir = Path('/content/travel-paper-superres')

if not input_path.exists():
    raise FileNotFoundError(input_path)
output_dir.mkdir(parents=True, exist_ok=True)

with Image.open(input_path) as image:
    print(f'Input: {input_path} | mode={image.mode} | size={image.size}')
    if image.size != EXPECTED_SIZE:
        raise ValueError(f'输入必须为 {EXPECTED_SIZE[0]}×{EXPECTED_SIZE[1]}，实际为 {image.size}')
    if 'A' not in image.mode:
        raise ValueError('输入必须包含独立 alpha 撕边；请使用纯纸面 world-paper.png')


In [ ]:
# @title 分块运行 RealESRGAN_x2plus
import cv2
import gc
import importlib
import numpy as np
from pathlib import Path
import sys
import torch
repo_dir = globals().get(
    'repo_dir',
    (Path('/content') if Path('/content').exists() else Path.cwd()) / 'Real-ESRGAN'
)
if not (repo_dir / 'realesrgan' / '__init__.py').exists():
    raise RuntimeError('Real-ESRGAN 仓库不存在；请先运行“安装 Real-ESRGAN”单元格。')
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
importlib.invalidate_caches()
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
model_path = repo_dir / 'weights' / 'RealESRGAN_x2plus.pth'
if not model_path.exists():
    raise RuntimeError('RealESRGAN_x2plus 模型不存在；请重新运行安装单元格。')

if not torch.cuda.is_available():
    raise RuntimeError('没有检测到 CUDA GPU。请在 Colab 中切换到 T4 GPU 后重新运行。')
print(f'GPU: {torch.cuda.get_device_name(0)}')

source_image = cv2.imread(str(input_path), cv2.IMREAD_UNCHANGED)
if source_image is None or source_image.ndim != 3 or source_image.shape[2] != 4:
    raise RuntimeError('无法以 BGRA 读取输入 PNG')
source_bgr = source_image[:, :, :3].copy()
source_alpha = source_image[:, :, 3].copy()
del source_image
gc.collect()

# 透明区域通常保存为黑色。先用纸张基色补齐，只把补齐结果交给 AI，
# 避免卷积在撕边内侧生成暗色光晕；最终 RGB 基线仍使用原始纸面。
alpha_weight = source_alpha.astype(np.float32)[:, :, None] / 255.0
paper_bgr = np.array([132, 182, 210], dtype=np.float32)
inference_bgr = np.clip(
    source_bgr.astype(np.float32) * alpha_weight + paper_bgr * (1.0 - alpha_weight),
    0, 255
).astype(np.uint8)
del alpha_weight

model = RRDBNet(
    num_in_ch=3, num_out_ch=3, num_feat=64,
    num_block=23, num_grow_ch=32, scale=2
)
upsampler = RealESRGANer(
    scale=2,
    model_path=str(model_path),
    model=model,
    tile=TILE_SIZE,
    tile_pad=TILE_PAD,
    pre_pad=0,
    half=True,
    gpu_id=0
)

ai_bgr, _ = upsampler.enhance(inference_bgr, outscale=UPSCALE)
del inference_bgr
gc.collect()
target_size = (EXPECTED_SIZE[0] * UPSCALE, EXPECTED_SIZE[1] * UPSCALE)
if (ai_bgr.shape[1], ai_bgr.shape[0]) != target_size:
    raise RuntimeError(f'超分尺寸异常: {ai_bgr.shape[1]}×{ai_bgr.shape[0]}')
if SAVE_RAW_AI:
    cv2.imwrite(str(output_dir / 'world-paper-16k-ai-raw.png'), ai_bgr, [cv2.IMWRITE_PNG_COMPRESSION, 4])
print(f'AI candidate ready: {target_size[0]}×{target_size[1]}')


In [ ]:
# @title 锁定低频，只叠加 AI 高频残差
import gc
import cv2
import numpy as np

base_bgr = cv2.resize(source_bgr, target_size, interpolation=cv2.INTER_LANCZOS4)
ai_low_8k = cv2.resize(ai_bgr, EXPECTED_SIZE, interpolation=cv2.INTER_AREA)
ai_low_16k = cv2.resize(ai_low_8k, target_size, interpolation=cv2.INTER_LANCZOS4)
del ai_low_8k

final_bgr = np.empty_like(base_bgr)
strip_height = 256
for y in range(0, target_size[1], strip_height):
    y2 = min(y + strip_height, target_size[1])
    base_strip = base_bgr[y:y2].astype(np.float32)
    residual = ai_bgr[y:y2].astype(np.float32) - ai_low_16k[y:y2].astype(np.float32)
    final_bgr[y:y2] = np.clip(base_strip + DETAIL_GAIN * residual, 0, 255).astype(np.uint8)
del ai_low_16k
gc.collect()

# 一次低频回写，让 16K 缩回 8K 后尽量贴近原始纸面。
roundtrip_bgr = cv2.resize(final_bgr, EXPECTED_SIZE, interpolation=cv2.INTER_AREA)
error_8k = source_bgr.astype(np.int16) - roundtrip_bgr.astype(np.int16)
for channel in range(3):
    correction = cv2.resize(error_8k[:, :, channel].astype(np.float32), target_size, interpolation=cv2.INTER_LANCZOS4)
    corrected = final_bgr[:, :, channel].astype(np.float32) + correction
    final_bgr[:, :, channel] = np.clip(corrected, 0, 255).astype(np.uint8)
    del correction, corrected
del error_8k, roundtrip_bgr
gc.collect()

alpha_16k = cv2.resize(source_alpha, target_size, interpolation=cv2.INTER_LANCZOS4)
final_bgra = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2BGRA)
final_bgra[:, :, 3] = alpha_16k
output_path = output_dir / OUTPUT_NAME
if not cv2.imwrite(str(output_path), final_bgra, [cv2.IMWRITE_PNG_COMPRESSION, 6]):
    raise RuntimeError('16K PNG 写入失败')
del final_bgra
gc.collect()
print(f'Constrained 16K paper: {output_path}')


In [ ]:
# @title 量化验证与局部对比图
import cv2
import json
import math
import numpy as np
from skimage.metrics import structural_similarity

roundtrip_bgr = cv2.resize(final_bgr, EXPECTED_SIZE, interpolation=cv2.INTER_AREA)
valid = source_alpha > 16
diff = source_bgr.astype(np.int16) - roundtrip_bgr.astype(np.int16)
valid_diff = diff[valid].astype(np.float32)
mae = float(np.mean(np.abs(valid_diff)))
mse = float(np.mean(valid_diff ** 2))
psnr = float('inf') if mse == 0 else 20.0 * math.log10(255.0 / math.sqrt(mse))

check_size = (2048, 1024)
source_check = cv2.resize(source_bgr, check_size, interpolation=cv2.INTER_AREA)
roundtrip_check = cv2.resize(roundtrip_bgr, check_size, interpolation=cv2.INTER_AREA)
alpha_check = cv2.resize(source_alpha, check_size, interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
paper_bgr = np.array([132, 182, 210], dtype=np.float32)
source_composite = np.clip(source_check * alpha_check[:, :, None] + paper_bgr * (1.0 - alpha_check[:, :, None]), 0, 255).astype(np.uint8)
roundtrip_composite = np.clip(roundtrip_check * alpha_check[:, :, None] + paper_bgr * (1.0 - alpha_check[:, :, None]), 0, 255).astype(np.uint8)
ssim = float(structural_similarity(source_composite, roundtrip_composite, channel_axis=2, data_range=255))
source_macro = cv2.GaussianBlur(source_composite, (0, 0), 6.0)
roundtrip_macro = cv2.GaussianBlur(roundtrip_composite, (0, 0), 6.0)
macro_mae = float(np.mean(np.abs(source_macro.astype(np.float32) - roundtrip_macro.astype(np.float32))))
alpha_roundtrip = cv2.resize(alpha_16k, EXPECTED_SIZE, interpolation=cv2.INTER_AREA)
alpha_mae = float(np.mean(np.abs(source_alpha.astype(np.float32) - alpha_roundtrip.astype(np.float32))))

thresholds = {
    'mae_max': 2.0,
    'psnr_min': 38.0,
    'ssim_min': 0.985,
    'macro_mae_max': 1.0,
    'alpha_mae_max': 1.0
}
passed = (
    mae <= thresholds['mae_max'] and
    psnr >= thresholds['psnr_min'] and
    ssim >= thresholds['ssim_min'] and
    macro_mae <= thresholds['macro_mae_max'] and
    alpha_mae <= thresholds['alpha_mae_max']
)
metrics = {
    'input': str(input_path),
    'output': str(output_path),
    'input_size': list(EXPECTED_SIZE),
    'output_size': list(target_size),
    'model': 'RealESRGAN_x2plus',
    'detail_gain': DETAIL_GAIN,
    'mae': mae,
    'psnr_db': psnr,
    'ssim_2048': ssim,
    'macro_mae_2048': macro_mae,
    'alpha_mae': alpha_mae,
    'thresholds': thresholds,
    'passed': passed
}
metrics_path = output_dir / 'paper-superres-metrics.json'
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))

preview = np.hstack([source_composite, roundtrip_composite])
cv2.putText(preview, 'Original 8K', (30, 58), cv2.FONT_HERSHEY_SIMPLEX, 1.25, (35, 28, 22), 3, cv2.LINE_AA)
cv2.putText(preview, '16K -> 8K', (check_size[0] + 30, 58), cv2.FONT_HERSHEY_SIMPLEX, 1.25, (35, 28, 22), 3, cv2.LINE_AA)
cv2.imwrite(str(output_dir / 'paper-roundtrip-preview.png'), preview)

def crop_at(image, fx, fy, size=768):
    height, width = image.shape[:2]
    x = min(max(int(width * fx - size / 2), 0), width - size)
    y = min(max(int(height * fy - size / 2), 0), height - size)
    return image[y:y + size, x:x + size].copy()

rows = []
for fx, fy in [(0.24, 0.28), (0.51, 0.50), (0.76, 0.68)]:
    row = np.hstack([
        crop_at(base_bgr, fx, fy),
        crop_at(ai_bgr, fx, fy),
        crop_at(final_bgr, fx, fy)
    ])
    cv2.putText(row, 'Lanczos', (18, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (35, 28, 22), 2, cv2.LINE_AA)
    cv2.putText(row, 'Raw AI', (786, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (35, 28, 22), 2, cv2.LINE_AA)
    cv2.putText(row, 'Constrained', (1554, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (35, 28, 22), 2, cv2.LINE_AA)
    rows.append(row)
cv2.imwrite(str(output_dir / 'paper-superres-crops.png'), np.vstack(rows), [cv2.IMWRITE_PNG_COMPRESSION, 6])

if not passed:
    print('WARNING: 回缩一致性未达到门槛。不要进入瓦片生成；先降低 DETAIL_GAIN 后重跑。')
else:
    print('PASS: 回缩一致性达到自动门槛，仍需人工检查局部纹理和慢速拖拽重复感。')


In [ ]:
# @title 打包结果
import shutil
from pathlib import Path

archive_base = (
    output_dir.parent / 'travel-paper-superres-result'
    if USE_GOOGLE_DRIVE
    else Path('/content/travel-paper-superres-result')
)
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=output_dir))
print(f'Archive: {archive_path} ({archive_path.stat().st_size / 1024 / 1024:.1f} MB)')
print('ZIP 已写入上述路径；其中必须包含 16K PNG、metrics JSON 和两张对比图。')

if AUTO_DOWNLOAD_ZIP and not USE_GOOGLE_DRIVE:
    from google.colab import files
    files.download(str(archive_path))
